# Как настроить BERT для классификации отзывов

На семинаре мы сравнили TF-IDF и замороженный BERT на отзывах. Теперь попробуем менять веса BERT, чтобы он лучше предсказывал класс рейтинга. Сначала повторим два знакомых способа, затем дообучим BERT и сравним все три на новых отзывах.

## 0. Данные и разбиение

In [1]:
!pip install -q "transformers>=4.45,<5" "scikit-learn>=1.4,<2" pandas scipy

Библиотеки уже установлены в проверочном окружении


In [2]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from IPython.display import display
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from tqdm.auto import tqdm
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, set_seed

/Users/arinanevolina/miniconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Устройство:", DEVICE)

Устройство: mps


In [4]:
WB_REVIEWS_URL = (
    "https://huggingface.co/datasets/Hplss/wb-review-dataset/"
    "resolve/main/filtered.csv"
)
LABEL_NAMES = np.array(["отрицательный", "нейтральный", "положительный"])

reviews = pd.read_csv(WB_REVIEWS_URL)
parts = reviews[["text", "pros", "cons"]].fillna("").astype(str)
reviews["input"] = parts.agg(" ".join, axis=1).str.strip()
reviews = reviews[reviews["input"].str.len() > 10].copy()
reviews["label"] = np.select(
    [reviews["rating"] <= 2, reviews["rating"] == 3], [0, 1], default=2
)
reviews = reviews.reset_index(drop=True)

display(reviews[["rating", "category_label", "input"]].head(3))
print("Всего отзывов:", len(reviews))

,rating,category_label,input
0,5,Электроника,"Звук не плохой, максимальная громкость высокая..."
1,5,Электроника,"Ничего. Подключалась 3 дня еле подключилась, н..."
2,5,Электроника,Сразу после первого подключения не работал вто...


Всего отзывов: 27248


Отзывы об одном товаре часто похожи, поэтому оставим все отзывы с одним `product_id` в одной части данных. На train обучаем модели. На validation сравниваем промежуточные результаты и выбираем лучшую эпоху BERT. На test проверяем все три способа после выбора.

In [5]:
def group_split(frame, test_size, seed):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    left, right = next(splitter.split(frame, groups=frame["product_id"]))
    return frame.iloc[left].copy(), frame.iloc[right].copy()


train_val, test_full = group_split(reviews, test_size=0.2, seed=42)
train_full, val_full = group_split(train_val, test_size=0.25, seed=43)
train, _ = train_test_split(train_full, train_size=6000, stratify=train_full["label"], random_state=42)
val, _ = train_test_split(val_full, train_size=1500, stratify=val_full["label"], random_state=43)
test, _ = train_test_split(test_full, train_size=2000, stratify=test_full["label"], random_state=44)

assert set(train["product_id"]).isdisjoint(val["product_id"])
assert set(train["product_id"]).isdisjoint(test["product_id"])
assert set(val["product_id"]).isdisjoint(test["product_id"])
print("train:", len(train), "val:", len(val), "test:", len(test))

train: 6000 val: 1500 test: 2000


In [6]:
balance = (
    train["label"].value_counts(normalize=True).sort_index()
    .rename(index=dict(enumerate(LABEL_NAMES)))
    .rename("доля в train")
)
display(balance.to_frame())

,доля в train
label,
отрицательный,0.163833
нейтральный,0.085833
положительный,0.750333


## 1. Первый вариант с TF-IDF

Сначала повторим знакомый способ. Признаки по словам помогают заметить фразы вроде «не подошёл», а признаки по символам могут связать опечатку с привычным написанием.

**Задача 1.** Постройте словные и символьные признаки для всех трёх частей данных. Оба словаря обучите на train. Преобразуйте validation и test с этими же словарями.

<details>
<summary><b>Нужна подсказка?</b></summary>

Нужны два независимых `TfidfVectorizer`: один для слов, второй с `analyzer="char_wb"`. Вспомните разницу между `fit_transform` и `transform` и не обучайте словари на validation или test.

</details>

In [7]:
word = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_features=40_000, sublinear_tf=True
)
char = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=2,
    max_features=40_000, sublinear_tf=True,
)
x_train_word = word.fit_transform(train["input"])
x_val_word = word.transform(val["input"])
x_test_word = word.transform(test["input"])
x_train_char = char.fit_transform(train["input"])
x_val_char = char.transform(val["input"])
x_test_char = char.transform(test["input"])

In [8]:
assert x_train_word.shape[0] == len(train)
assert x_val_char.shape[0] == len(val)
assert x_test_char.shape[0] == len(test)
print(x_train_word.shape, x_train_char.shape)

(6000, 26801) (6000, 40000)


**Задача 2.** Соедините два набора признаков и обучите на них одну логистическую регрессию. Оцените её на validation по accuracy и macro-F1.

<details>
<summary><b>Нужна подсказка?</b></summary>

Разреженные признаки объединяются через `hstack`. После этого используйте обычные `fit` и `predict` у `LogisticRegression`. Проверьте порядок строк во всех матрицах.

</details>

In [9]:
x_train = hstack([x_train_word, x_train_char]).tocsr()
x_val = hstack([x_val_word, x_val_char]).tocsr()
x_test = hstack([x_test_word, x_test_char]).tocsr()

tfidf_clf = LogisticRegression(max_iter=500, class_weight="balanced", C=4.0)
tfidf_clf.fit(x_train, train["label"])
pred_val_tfidf = tfidf_clf.predict(x_val)
print("val accuracy:", accuracy_score(val["label"], pred_val_tfidf))
print("val macro-F1:", f1_score(val["label"], pred_val_tfidf, average="macro"))

val accuracy: 0.8306666666666667
val macro-F1: 0.6294311387365116


## 2. Замороженный BERT

Заменим TF-IDF на эмбеддинги BERT, как на семинаре. Сначала напишите `mean_pool`, чтобы получить один эмбеддинг на отзыв. При усреднении пропускайте `[PAD]`. Затем обучите логистическую регрессию на этих эмбеддингах. Веса BERT на этом шаге остаются прежними.

In [10]:
BERT_ID = "cointegrated/rubert-tiny2"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_ID)
bert_encoder = AutoModel.from_pretrained(BERT_ID).to(DEVICE).eval()

<details>
<summary><b>Нужна подсказка?</b></summary>

Сначала сделайте форму attention mask совместимой с hidden states. Затем padding должен дать нулевой вклад и в сумму, и в число токенов. Пригодятся `unsqueeze`, `sum` и `clamp`.

</details>

In [11]:
def mean_pool(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1)
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

In [12]:
def embed_texts(texts, batch_size=None, max_length=128):
    if batch_size is None:
        batch_size = 128 if DEVICE.type == "cuda" else 32
    vectors = []
    for start in tqdm(range(0, len(texts), batch_size), leave=False):
        batch = bert_tokenizer(
            texts[start:start + batch_size], padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        ).to(DEVICE)
        with torch.inference_mode():
            hidden = bert_encoder(**batch).last_hidden_state
        vectors.append(F.normalize(mean_pool(hidden, batch["attention_mask"]), dim=1).cpu())
    return torch.cat(vectors).numpy()


train_emb = embed_texts(train["input"].tolist())
val_emb = embed_texts(val["input"].tolist())
test_emb = embed_texts(test["input"].tolist())

  0%|          | 0/188 [00:00<?, ?it/s]

  1%|          | 1/188 [00:00<00:39,  4.79it/s]

  2%|▏         | 4/188 [00:00<00:13, 13.57it/s]

  4%|▎         | 7/188 [00:00<00:10, 17.80it/s]

  6%|▋         | 12/188 [00:00<00:07, 23.55it/s]

 10%|▉         | 18/188 [00:00<00:05, 32.71it/s]

... (30 outputs left out)

 41%|████▏     | 26/63 [00:00<00:00, 124.69it/s]

 62%|██████▏   | 39/63 [00:00<00:00, 85.44it/s] 

 78%|███████▊  | 49/63 [00:00<00:00, 82.69it/s]

 92%|█████████▏| 58/63 [00:00<00:00, 80.11it/s]

<details>
<summary><b>Нужна подсказка?</b></summary>

BERT здесь уже ничего не обучает: `train_emb` и `val_emb` — готовые признаки. Нужны `LogisticRegression`, `accuracy_score` и `f1_score(..., average="macro")`.

</details>

In [13]:
frozen_clf = LogisticRegression(max_iter=500, class_weight="balanced", C=4.0)
frozen_clf.fit(train_emb, train["label"])
pred_val_frozen = frozen_clf.predict(val_emb)
print("val accuracy:", accuracy_score(val["label"], pred_val_frozen))
print("val macro-F1:", f1_score(val["label"], pred_val_frozen, average="macro"))

val accuracy: 0.73
val macro-F1: 0.5918024466326472


## 3. Дообучаем BERT

Теперь будем обучать сам BERT. На каждом шаге он получает батч отзывов и правильные классы. Начнём с `ReviewDataset`, который выдаёт по одному примеру.

**Задача.** По индексу верните токены одного отзыва и его класс под ключом `labels`. Слишком длинный отзыв обрежьте до `max_length` токенов. До общей длины дополним тексты при сборке батча.

[Документация Dataset и DataLoader](https://pytorch.org/docs/stable/data.html)

<details>
<summary><b>Нужна подсказка?</b></summary>

`Dataset` должен уметь ответить на два вопроса: сколько в нём объектов и как получить один объект по индексу. Токенизируйте один текст без padding и добавьте его числовую метку под ключом `labels`.

</details>

In [14]:
from torch.utils.data import DataLoader, Dataset

class ReviewDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=128):
        self.texts = frame["input"].tolist()
        self.labels = frame["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        item = self.tokenizer(self.texts[index], truncation=True, max_length=self.max_length)
        item["labels"] = self.labels[index]
        return item

In [15]:
train_dataset = ReviewDataset(train, bert_tokenizer)
example = train_dataset[0]
assert "input_ids" in example and "labels" in example
print("Объектов:", len(train_dataset), "токенов в первом:", len(example["input_ids"]))

Объектов: 6000 токенов в первом: 59


При сборке батча дополняем короткие отзывы до длины самого длинного отзыва в этом батче. Например, если в батче отзывы на 5 и 12 токенов, к первому добавим 7 токенов `[PAD]`. В другом батче максимум может быть другим. Поэтому такой padding называют динамическим.

`DataLoader` берёт из `Dataset` очередную группу примеров и передаёт её `collator`. Collator собирает из этих примеров батч тензоров. В нашем коде `DataCollatorWithPadding` вызывает `tokenizer.pad`, который добавляет токены `[PAD]`.

**Задача.** Соберите три `DataLoader` и передайте каждому готовый `collator`. Перемешивайте отзывы в train, а в validation и test сохраняйте исходный порядок.

[Документация DataCollatorWithPadding](https://huggingface.co/docs/transformers/main_classes/data_collator)

<details>
<summary><b>Нужна подсказка?</b></summary>

Передайте созданный `ReviewDataset` в `DataLoader`. За padding отвечает аргумент `collate_fn`, а перемешивание нужно только обучающей выборке.

</details>

In [16]:
collator = DataCollatorWithPadding(bert_tokenizer, return_tensors="pt")

def make_loader(frame, batch_size, shuffle):
    dataset = ReviewDataset(frame, bert_tokenizer)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collator
    )


train_loader = make_loader(train, 32 if DEVICE.type == "cuda" else 8, True)
val_loader = make_loader(val, 64 if DEVICE.type == "cuda" else 16, False)
test_loader = make_loader(test, 64 if DEVICE.type == "cuda" else 16, False)

In [17]:
first_batch = next(iter(train_loader))
print({name: tuple(tensor.shape) for name, tensor in first_batch.items()})
assert first_batch["input_ids"].shape == first_batch["attention_mask"].shape

{'input_ids': (8, 101), 'token_type_ids': (8, 101), 'attention_mask': (8, 101), 'labels': (8,)}


Теперь, когда батч готов, нужна модель, которая поставит каждому отзыву три оценки (`logits`), по одной на класс. Для этого поверх BERT добавим небольшой обучаемый слой. Его называют классификационной головой.

При загрузке модель может предупредить, что у этого слоя случайные веса. Слой создан для нашей задачи и ещё не обучен. Оптимизатор AdamW будет обновлять веса модели.

**Задача.** Загрузите модель на три класса и настройте AdamW с `lr=2e-5`.

<details>
<summary><b>Нужна подсказка?</b></summary>

Модель создаётся через `AutoModelForSequenceClassification.from_pretrained`; число классов и словари названий передаются при загрузке. Optimizer — `torch.optim.AdamW` по параметрам модели.

</details>

In [18]:
id2label = {i: name for i, name in enumerate(LABEL_NAMES)}
label2id = {name: i for i, name in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    BERT_ID, num_labels=3, id2label=id2label, label2id=label2id,
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


После каждой эпохи будем проверять, стало ли лучше на validation. Для этого нужны оценки модели, правильные классы и средний loss — число, которое показывает, насколько оценки расходятся с правильными ответами. Из оценок потом получим accuracy и macro-F1.

**Задача.** Напишите `collect_predictions`. Обойдите батчи из переданного `loader` с текущими весами модели. Верните оценки модели (`logits`), правильные классы и средний loss на отзыв.

<details>
<summary><b>Нужна подсказка?</b></summary>

Для проверки нужны `model.eval()` и `torch.inference_mode()`. Сохраняйте logits и labels на CPU, а в конце объедините части через `torch.cat`. Средний loss считайте по объектам, а не по числу батчей.

</details>

In [19]:
def collect_predictions(model, loader):
    model.eval()
    all_logits = []
    all_labels = []
    total_loss = 0.0
    total_objects = 0

    with torch.inference_mode():
        for batch in loader:
            batch = {name: tensor.to(DEVICE) for name, tensor in batch.items()}
            output = model(**batch)
            size = batch["labels"].size(0)
            total_loss += output.loss.item() * size
            total_objects += size
            all_logits.append(output.logits.cpu())
            all_labels.append(batch["labels"].cpu())

    return torch.cat(all_logits), torch.cat(all_labels), total_loss / total_objects

In [20]:
def metrics_from_logits(logits, labels, loss):
    predictions = logits.argmax(dim=1).numpy()
    probabilities = logits.softmax(dim=1).numpy()
    labels = labels.numpy()
    return {
        "loss": loss,
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "predictions": predictions,
        "probabilities": probabilities,
        "targets": labels,
    }

Один проход по всем обучающим батчам называют эпохой. В каждом батче модель предсказывает классы, считает loss по известным ответам и обновляет веса.

**Задача.** Напишите `train_one_epoch`. Обучите модель на всех батчах из `loader` и верните средний loss на отзыв за эпоху.

<details>
<summary><b>Нужна подсказка?</b></summary>

В режиме обучения для каждого батча нужно очистить старые градиенты, получить loss, посчитать новые градиенты и обновить параметры. Вспомните методы `zero_grad`, `backward` и `step`.

</details>

In [21]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total_objects = 0

    for batch in tqdm(loader, leave=False):
        batch = {name: tensor.to(DEVICE) for name, tensor in batch.items()}
        optimizer.zero_grad()
        output = model(**batch)
        output.loss.backward()
        optimizer.step()
        size = batch["labels"].size(0)
        total_loss += output.loss.item() * size
        total_objects += size

    return total_loss / total_objects

После двух эпох результат может быть хуже, чем после первой. Поэтому после каждой эпохи будем считать macro-F1 на validation и сохранять веса, которые дали лучший результат. `state_dict` хранит состояние модели, включая веса.

**Задача.** Запустите две эпохи. После каждой посчитайте macro-F1 на validation. В конце загрузите в модель сохранённое лучшее состояние.

<details>
<summary><b>Нужна подсказка?</b></summary>

После каждой эпохи используйте уже написанные функции обучения и проверки. При улучшении validation macro-F1 сохраните копию `state_dict`; здесь пригодится `copy.deepcopy`. В конце верните лучшее состояние через `load_state_dict`.

</details>

In [22]:
EPOCHS = 2
best_macro_f1 = -1.0
best_state = None

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_logits, val_labels, val_loss = collect_predictions(model, val_loader)
    val_metrics = metrics_from_logits(val_logits, val_labels, val_loss)
    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        best_state = copy.deepcopy(model.state_dict())
    print({
        "epoch": epoch + 1, "train_loss": train_loss,
        "val_loss": val_loss, "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
    })

model.load_state_dict(best_state)
print("Лучший validation macro-F1:", best_macro_f1)

  0%|          | 0/750 [00:00<?, ?it/s]

  0%|          | 1/750 [00:02<28:34,  2.29s/it]

  0%|          | 2/750 [00:02<12:48,  1.03s/it]

  0%|          | 3/750 [00:02<08:16,  1.51it/s]

  1%|          | 5/750 [00:02<04:32,  2.73it/s]

  1%|          | 6/750 [00:03<03:53,  3.18it/s]

... (332 outputs left out)

 98%|█████████▊| 737/750 [00:17<00:00, 46.09it/s]

 99%|█████████▉| 742/750 [00:17<00:00, 45.29it/s]

100%|█████████▉| 747/750 [00:17<00:00, 45.65it/s]

{'epoch': 2, 'train_loss': 0.39602269998192785, 'val_loss': 0.39985486141840615, 'val_accuracy': 0.8426666666666667, 'val_macro_f1': 0.5482895463405871}
Лучший validation macro-F1: 0.5482895463405871


## 4. Сравнение на test

На validation мы выбрали эпоху BERT. Теперь проверим три готовые модели на одной и той же отложенной выборке.

**Задача.** Получите предсказания на test от TF-IDF, замороженного BERT и дообученного BERT. Покажите в одной таблице их accuracy и macro-F1.

<details>
<summary><b>Нужна подсказка?</b></summary>

Для двух sklearn-моделей нужен `predict`, для BERT — `collect_predictions` и `metrics_from_logits`. Не вызывайте никакой `fit` на test.

</details>

In [23]:
pred_test_tfidf = tfidf_clf.predict(x_test)
pred_test_frozen = frozen_clf.predict(test_emb)
test_logits, test_labels, test_loss = collect_predictions(model, test_loader)
finetuned_test = metrics_from_logits(test_logits, test_labels, test_loss)

final_results = pd.DataFrame([
    ["TF-IDF", accuracy_score(test["label"], pred_test_tfidf), f1_score(test["label"], pred_test_tfidf, average="macro")],
    ["замороженный BERT", accuracy_score(test["label"], pred_test_frozen), f1_score(test["label"], pred_test_frozen, average="macro")],
    ["дообученный BERT", finetuned_test["accuracy"], finetuned_test["macro_f1"]],
], columns=["способ", "accuracy", "macro_f1"])
display(final_results.round(3))

,способ,accuracy,macro_f1
0,TF-IDF,0.814,0.605
1,замороженный BERT,0.707,0.563
2,дообученный BERT,0.836,0.535


Accuracy показывает, какая доля отзывов классифицирована правильно. Положительных отзывов больше всего, и они сильнее влияют на это число. Macro-F1 даёт каждому классу одинаковый вес.

В логистической регрессии мы включили `class_weight="balanced"`. При этом ошибки на редких классах получают больший вес. При дообучении BERT у каждого отзыва был одинаковый вес. Посмотрим, на каких классах это могло сказаться.

## 5. Анализ ошибок

В итоговой таблице по два числа на модель. Чтобы понять, какие классы путает дообученный BERT, построим матрицу ошибок. По строкам будут правильные классы, по столбцам — прогнозы модели. На диагонали окажутся правильные ответы.

**Задача.** Постройте эту матрицу и найдите десять ошибок, в которых модель была наиболее уверена. В коде уверенность — вероятность, которую модель присвоила выбранному классу. Посмотрите, куда чаще всего попали нейтральные отзывы.

<details>
<summary><b>Нужна подсказка?</b></summary>

Таблицу «правильный класс × предсказанный класс» можно получить через `pd.crosstab`. Уверенные ошибки — это неверные строки после сортировки по максимальной вероятности.

</details>

In [24]:
analysis = test[["rating", "input", "label"]].copy()
analysis["fine-tuned BERT"] = finetuned_test["predictions"]
analysis["уверенность"] = finetuned_test["probabilities"].max(axis=1)

confusion = pd.crosstab(
    LABEL_NAMES[analysis["label"]], LABEL_NAMES[analysis["fine-tuned BERT"]],
    rownames=["правильный класс"], colnames=["предсказание"],
)
display(confusion)
mistakes = analysis[analysis["fine-tuned BERT"] != analysis["label"]]
display(mistakes.sort_values("уверенность", ascending=False).head(10))

предсказание,нейтральный,отрицательный,положительный
правильный класс,,,
нейтральный,0,79,85
отрицательный,0,268,75
положительный,1,89,1403


,rating,input,label,fine-tuned BERT,уверенность
26908,1,"Очень вкусно. Орешков очень много, как щербет ...",0,2,0.988766
4998,3,"Платье сарафан, как раз село на мою уже нескол...",1,2,0.985125
10470,3,"Очень красивые, самый необходимый обьем, приго...",1,2,0.983250
15477,2,Всем доброе ! Велосипед колеса 24 я сегодня по...,0,2,0.982105
9262,1,"Пересушил мне все волосы, кератином там даже н...",0,2,0.978121
13571,3,"Пазлы хорошие, красивые, достаточно сложные. П...",1,2,0.977826
5943,2,"Приятный , хорошо увлажнял Пользовалась неделю...",0,2,0.976958
6483,3,"пользуюсь месяц и могу сказать , что он не защ...",1,2,0.975751
6272,3,"По качеству средне, в целом не плох, но у меня...",1,2,0.973878
25001,2,"Красивая, удобная, не течет Но тепло держит ма...",0,2,0.971968


### Проверим модель на своих отзывах

На test мы сравнили предсказания с классами, которые получили из звёзд. Теперь представьте, что пришёл новый отзыв и у нас есть только его текст. Какой класс предскажет модель, если автор хвалит товар, но ругает доставку?

**Задача.** Напишите `predict_texts`. На входе список отзывов, на выходе по одному номеру класса для каждого отзыва. Проверьте функцию на `custom_reviews`. Можно добавить свои примеры. Покажите таблицу с текстами и названиями предсказанных классов и сравните прогнозы со своей оценкой.

<details>
<summary><b>Нужна подсказка?</b></summary>

Токенизируйте список отзывов одним батчем и передайте его модели. Для предсказания пригодятся `model.eval()` и `torch.inference_mode()`, как при проверке на test. Для каждого текста модель выдаёт три оценки (`logits`), по одной на класс. `argmax` вернёт номер класса с самой высокой оценкой.

</details>

In [25]:
custom_reviews = [
    "не могу не рекомендовать", "качество огонь, доставка ужас",
    "ну такое", "товар имба, закажу ещё", "оооочень довольна!!!",
]

def predict_texts(texts):
    batch = bert_tokenizer(
        texts, padding=True, truncation=True, max_length=128, return_tensors="pt"
    ).to(DEVICE)
    model.eval()
    with torch.inference_mode():
        return model(**batch).logits.argmax(dim=1).cpu().numpy()


custom_pred = predict_texts(custom_reviews)
display(pd.DataFrame({"текст": custom_reviews, "класс": LABEL_NAMES[custom_pred]}))

,текст,класс
0,не могу не рекомендовать,отрицательный
1,"качество огонь, доставка ужас",отрицательный
2,ну такое,отрицательный
3,"товар имба, закажу ещё",положительный
4,оооочень довольна!!!,положительный


## Итог

В контрольном запуске дообученный BERT получил самую высокую accuracy. По macro-F1 первым оказался TF-IDF. Матрица ошибок показывает, что дообученный BERT плохо распознавал редкий нейтральный класс. В бонусе можно проверить, поможет ли больший вес для редких классов при обучении.

In [26]:
SAVE_DIR = "rubert-reviews"
model.save_pretrained(SAVE_DIR)
bert_tokenizer.save_pretrained(SAVE_DIR)
print("Модель сохранена в", SAVE_DIR)

Модель сохранена в rubert-reviews


## Бонус для дома

1. Загрузите сохранённую модель и проверьте предсказания.
2. Посмотрите, какая часть длинного отзыва останется при обрезке до 64, 128 и 256 токенов. Это называется truncation.
3. Попробуйте дать ошибкам на редких классах больший вес при обучении. Сравните macro-F1 и матрицы ошибок на validation.

<details>
<summary><b>Нужна подсказка?</b></summary>

И токенизатор, и модель загружаются методом `from_pretrained` из одной папки. Совпадение двух массивов предсказаний можно проверить через `np.array_equal`.

</details>

In [27]:
reloaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(DEVICE).eval()
batch = reloaded_tokenizer(
    custom_reviews, padding=True, truncation=True, max_length=128, return_tensors="pt"
).to(DEVICE)
with torch.inference_mode():
    reloaded_pred = reloaded_model(**batch).logits.argmax(dim=1).cpu().numpy()
print("Предсказания совпали:", np.array_equal(custom_pred, reloaded_pred))

Предсказания совпали: True


<details>
<summary><b>Нужна подсказка?</b></summary>

Получите `input_ids` с нужным `max_length`, а затем превратите их обратно в текст методом `decode`. Повторите это для одного и того же длинного отзыва.

</details>

In [28]:
token_lengths = test["input"].map(
    lambda text: len(bert_tokenizer(text, truncation=False)["input_ids"])
)
long_text = test.loc[token_lengths.idxmax(), "input"]

for max_length in [64, 128, 256]:
    ids = bert_tokenizer(long_text, truncation=True, max_length=max_length)["input_ids"]
    print(f"max_length={max_length}")
    print(bert_tokenizer.decode(ids, skip_special_tokens=True), "\n")

max_length=64
Очень разочарованы Доставка быстрая И это всё, что понравилось!!!! Велосипед был приобретëн в подарок. Единственный плюс при выборе товара, являлось наличие корзины (!!! ) а еë НЕТ!!! Я СЧИТАЮ, ЧТО ТОВАР НЕ В 

max_length=128
Очень разочарованы Доставка быстрая И это всё, что понравилось!!!! Велосипед был приобретëн в подарок. Единственный плюс при выборе товара, являлось наличие корзины (!!! ) а еë НЕТ!!! Я СЧИТАЮ, ЧТО ТОВАР НЕ В ПОЛНОМ СООТВЕТСТВИИ!!! ЛИБО НА ПВЗ СКРЫЛИ ПРИСУТСТВИЕ КОРЗИНЫ РАЗБИРАЙТЕСЬ!!! ЖДУ ВАШЕГО ОТВЕТА Велосипед был приобретëн в подарок 

max_length=256
Очень разочарованы Доставка быстрая И это всё, что понравилось!!!! Велосипед был приобретëн в подарок. Единственный плюс при выборе товара, являлось наличие корзины (!!! ) а еë НЕТ!!! Я СЧИТАЮ, ЧТО ТОВАР НЕ В ПОЛНОМ СООТВЕТСТВИИ!!! ЛИБО НА ПВЗ СКРЫЛИ ПРИСУТСТВИЕ КОРЗИНЫ РАЗБИРАЙТЕСЬ!!! ЖДУ ВАШЕГО ОТВЕТА Велосипед был приобретëн в подарок. Единственный плюс при выборе товара, являлось наличие корзины 

### Как дать редким классам больший вес

```python
counts = train["label"].value_counts().sort_index().to_numpy()
weights = len(train) / (len(LABEL_NAMES) * counts)
weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

output = model(**batch)
loss = F.cross_entropy(output.logits, batch["labels"], weight=weights)
```

Сравнивать варианты нужно по validation macro-F1. Test снова открывается только после выбора окончательного способа.

Полезные ссылки:

- [Классификация текста в Transformers](https://huggingface.co/docs/transformers/tasks/sequence_classification)
- [Dataset и DataLoader](https://pytorch.org/docs/stable/data.html)
- [AdamW](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html)
- [DataCollatorWithPadding](https://huggingface.co/docs/transformers/main_classes/data_collator)
- [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)